# PRISM factorial generation on Colab (vLLM)

Runs the 6-cell factorial (A, A_prime, A_instr, B_minus, B_rnt, B) x queries x runs against any vLLM-loadable model, via `generation/run_prompts_colab.py`. Default parameters below reproduce the released `llama3.1-8b.jsonl` setup (vanilla Llama-3.1-8B-Instruct, `borderline_50.json`) as a pipeline sanity check -- edit the **Parameters** cell to point at a different model/adapter/dataset (e.g. an EM organism + `harmbench_50.json`) for the reverse-alignment-faking run.

**Before running:**
1. `Runtime > Change runtime type > T4 GPU`.
2. Key icon (Secrets) in the left sidebar: add `ANTHROPIC_API_KEY` (always required, for the judge). Add `HF_TOKEN` too if `MODEL`/`LORA_ADAPTER` below point at a gated repo (e.g. `meta-llama/Llama-3.1-8B-Instruct`, needed for the EM organism adapters) -- not required for the public AWQ model used by the default verification run.
3. Enable notebook access for each secret you add.

In [ ]:
!pip install -q vllm anthropic huggingface_hub

In [ ]:
!git clone https://github.com/AyishaR/prism-alignment-faking.git
%cd prism-alignment-faking

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
try:
    hf_token = userdata.get("HF_TOKEN")
    from huggingface_hub import login
    login(hf_token)
except Exception:
    print("no HF_TOKEN secret set -- fine unless MODEL/LORA_ADAPTER below are gated")

In [ ]:
# --- Parameters: edit these to switch model / adapter / dataset / prompt wording ---

MODEL = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"  # base model vLLM loads
LORA_ADAPTER = None   # e.g. "ModelOrganismsForEM/Llama-3.1-8B-Instruct_risky-financial-advice"
QUANTIZATION = "awq"  # "none" when using a full-precision base (e.g. with a LoRA adapter)
TAG = "llama3.1-8b_verify"   # output file: data/generations/factorial/<TAG>.jsonl
QUERIES = "borderline_50.json"   # filename under queries/, e.g. "harmbench_50.json"
PROMPTS_DIR = "factorial"   # subfolder under prompts/ with <cell>.txt files
CELLS = ["A", "A_prime", "A_instr", "B_minus", "B_rnt", "B"]
RUNS = 10

In [ ]:
import subprocess

cmd = ["python", "generation/run_prompts_colab.py", "--model", MODEL, "--tag", TAG,
       "--quantization", QUANTIZATION, "--queries", QUERIES, "--prompts-dir", PROMPTS_DIR,
       "--cells", *CELLS, "--runs", str(RUNS)]
if LORA_ADAPTER:
    cmd += ["--lora-adapter", LORA_ADAPTER]
print("running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from google.colab import files

zip_name = f"{TAG}.zip"
!zip -j {zip_name} data/generations/factorial/{TAG}.jsonl
files.download(zip_name)